### Helper Notebook: Build a Compact GloVe Subset (for KNN & LSH from scratch)

#### Purpose
This mini-notebook prepares a tiny, dense word-embedding pack tailored to our small tweet dataset. The goal is to keep the main KNN + LSH (NumPy-only) notebook focused on the algorithms (exact KNN and approximate search via LSH) without requiring users to download or load the full GloVe 6B corpus.

#### What this notebook does
- **Tokenizes** the 24 tweets using our existing utilities (`clean_text`, `tokenize`) to ensure preprocessing is consistent with the main notebook.
- **Streams** `glove.6B.300d.txt` **once** and extracts vectors **only** for tokens that actually appear in our corpus (memory-light and fast).
- Builds a **deterministic subset vocabulary** with:
  - `"<unk>"` at index **0**,
  - all remaining tokens in **alphabetical** order (simple, reproducible).
- Constructs a compact **(V × 300)** embedding matrix:
  - found tokens → their GloVe vectors,
  - OOV tokens → the **mean vector** of all found tokens (robust, stable scale).
- Saves three small artifacts to `./data`:
  - `glove_subset_vocab.json` — token → index
  - `glove_subset.npy` — float32 embedding matrix
  - `glove_subset_meta.json` — coverage & stats

#### Why this helps KNN & LSH
The main notebook needs dense document embeddings to:
- run **brute-force KNN** with cosine similarity as a correctness baseline, and
- implement **Locality Sensitive Hashing (random hyperplanes)** for **fast approximate search** with speed/recall trade-offs.

#### Reproducibility & Notes
- Preprocessing matches the main notebook by importing `clean_text` and `tokenize` from `utils_NLP.py`.
- Vocabulary order is deterministic (alphabetical; `"<unk>"` at 0).
- OOV handling uses the **mean of found vectors** (consistent fallback).
- The notebook prints **relative paths** (`./data/<file>`) for saved artifacts.

In [1]:
from pathlib import Path
import json
from collections import Counter
import sys
import numpy as np

In [2]:
TOOLS_DIR = Path("..") / "tools"   # from /data → ../tools
if not TOOLS_DIR.exists():
    raise FileNotFoundError(f"Expected tools folder at: {TOOLS_DIR.resolve()}")

In [3]:
# Prepend tools to sys.path
sys.path.insert(0, str(TOOLS_DIR.resolve()))

# Now import your utils
from utils_NLP import clean_text, tokenize

In [4]:
# JSON Paths
DATA_JSON = Path("./texts_5labels.json")

In [5]:
with DATA_JSON.open("r", encoding="utf-8") as f:
    data = json.load(f)

texts = data["texts"]
labels = data["labels"]

In [6]:
tokenized_texts = [tokenize(clean_text(t)) for t in texts]

In [7]:
print(tokenized_texts[:3])

[['ai', 'models', 'from', 'scratch', 'are', 'fun', 'to', 'build', 'with', 'numpy'], ['training', 'neural', 'nets', 'takes', 'time', 'but', "it's", 'rewarding'], ['debugging', 'python', 'code', 'late', 'at', 'night', 'hits', 'different']]


In [8]:
# Collect the set of tokens we need from GloVe
tokens_needed = {tok for toks in tokenized_texts for tok in toks}

print(f"Docs: {len(texts)} | Unique tokens needed: {len(tokens_needed)}")
print("First 3 tokenized docs:")
for i in range(3):
    print(f"[{i}] -> {tokenized_texts[i]}")


Docs: 24 | Unique tokens needed: 160
First 3 tokenized docs:
[0] -> ['ai', 'models', 'from', 'scratch', 'are', 'fun', 'to', 'build', 'with', 'numpy']
[1] -> ['training', 'neural', 'nets', 'takes', 'time', 'but', "it's", 'rewarding']
[2] -> ['debugging', 'python', 'code', 'late', 'at', 'night', 'hits', 'different']


In [9]:
#Extract only needed tokens from local GloVe (glove.6B.300d.txt in /data)

EMBED_DIM = 300
EMB_PATH = Path("./glove.6B.300d.txt")  # file is next to this notebook

if not EMB_PATH.exists():
    raise FileNotFoundError(f"GloVe file not found at: {EMB_PATH.resolve()}")

In [10]:
emb = {}
kept = 0
needed = len(tokens_needed)

with EMB_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        parts = line.rstrip("\n").split()  # robust split on any whitespace
        if not parts:
            continue
        word = parts[0]
        if word in tokens_needed:
            vec = np.asarray(parts[1:], dtype=np.float32)
            if vec.size == EMBED_DIM:
                emb[word] = vec
                kept += 1
                # Optional short-circuit if we've found all tokens
                if kept == needed:
                    break

In [11]:
coverage = kept / max(1, needed)
print(f"GloVe coverage: {kept}/{needed} tokens ({coverage:.1%})")
# Peek a couple of entries
for i, (w, v) in enumerate(emb.items()):
    if i >= 3: break
    print(f"{w:>12s} -> shape {v.shape}, first 3 dims: {v[:3]}")


GloVe coverage: 156/160 tokens (97.5%)
         the -> shape (300,), first 3 dims: [ 0.04656    0.21318   -0.0074364]
          of -> shape (300,), first 3 dims: [-0.076947 -0.021211  0.21271 ]
          to -> shape (300,), first 3 dims: [-0.25756  -0.057132 -0.6719  ]


In [12]:
# Deterministic vocab order: alphabetical only
sorted_tokens = sorted(tokens_needed)

stoi = {"<unk>": 0}
itos = ["<unk>"] + sorted_tokens
# build stoi from itos
for idx, tok in enumerate(sorted_tokens, start=1):
    stoi[tok] = idx

V = len(stoi)
E = np.zeros((V, EMBED_DIM), dtype=np.float32)

In [13]:
# Build & save the compact GloVe subset

# 1) Fallback vector: mean of all FOUND token vectors
found_vecs = [emb[t] for t in sorted_tokens if t in emb]
if not found_vecs:
    raise ValueError("No overlap between tokens_needed and GloVe. Check tokenization or casing.")
mean_vec = np.vstack(found_vecs).mean(axis=0).astype(np.float32)


In [14]:
# 2) Fill E: <unk> at 0, OOV -> mean_vec
E[0] = mean_vec
missing = []
for tok in sorted_tokens:
    idx = stoi[tok]
    vec = emb.get(tok)
    if vec is None:
        E[idx] = mean_vec
        missing.append(tok)
    else:
        E[idx] = vec

coverage = (len(sorted_tokens) - len(missing)) / max(1, len(sorted_tokens))
coverage


0.975

In [15]:
# 3) Save files in ./data (current folder)
VOCAB_PATH = Path("./glove_subset_vocab.json")
EMB_OUT_PATH = Path("./glove_subset.npy")
META_PATH  = Path("./glove_subset_meta.json")

with VOCAB_PATH.open("w", encoding="utf-8") as f:
    json.dump(stoi, f, ensure_ascii=False, indent=2)

np.save(EMB_OUT_PATH, E)

meta = {
    "embed_dim": EMBED_DIM,
    "vocab_size": len(stoi),
    "tokens_needed": len(sorted_tokens),
    "found": len(sorted_tokens) - len(missing),
    "missing": len(missing),
    "coverage": coverage,
    "examples_missing": missing[:10],
}
with META_PATH.open("r+" if META_PATH.exists() else "w", encoding="utf-8") as f:
    # overwrite if exists
    f.seek(0); f.truncate(0)
    json.dump(meta, f, ensure_ascii=False, indent=2)
    
print("Saved:")
print(f" - ./{VOCAB_PATH.name}")
print(f" - ./{EMB_OUT_PATH.name} {E.shape} {E.dtype}")
print(f" - ./{META_PATH.name}")
print(f"Coverage: {meta['found']}/{meta['tokens_needed']} ({coverage:.1%})")



Saved:
 - ./glove_subset_vocab.json
 - ./glove_subset.npy (161, 300) float32
 - ./glove_subset_meta.json
Coverage: 156/160 (97.5%)
